# Bayesian Inversion via MCMC

In [ ]:
import emcee
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(" STEP 1: DEFINE TARGET PROPERTIES & PARAMETER BOUNDS ")
target_ucs_mean = 90e6
cov = 0.10
target_ucs_std = target_ucs_mean * cov

print(f"Target Mean Peak UCS : {target_ucs_mean} Pa")
print(f"Target Std Deviation : {target_ucs_std} Pa")

input_cols = list(X_train.columns)
bounds = [(X_train[col].min(), X_train[col].max()) for col in input_cols]

print("\n STEP 2: DEFINE BAYESIAN MCMC FUNCTIONS ")

def log_prior(theta, bounds):
    for val, (min_val, max_val) in zip(theta, bounds):
        if not (min_val <= val <= max_val):
            return -np.inf
    return 0.0

def log_likelihood(theta, target_mean, target_std, model, scaler_X, scaler_y):
    theta_scaled = scaler_X.transform([theta])
    pred_scaled = model.predict(theta_scaled)
    pred_orig = scaler_y.inverse_transform(pred_scaled).flatten()
    predicted_ucs = pred_orig[0]
    return -0.5 * ((predicted_ucs - target_mean) / target_std)**2

def log_probability(theta, bounds, target_mean, target_std, model, scaler_X, scaler_y):
    lp = log_prior(theta, bounds)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, target_mean, target_std, model, scaler_X, scaler_y)

print("\n STEP 3: RUN MCMC (METROPOLIS-HASTINGS) ")
ndim = len(input_cols)
nwalkers = 32
nsteps = 1000

pos = np.zeros((nwalkers, ndim))
for i in range(ndim):
    min_val, max_val = bounds[i]
    pos[:, i] = np.random.uniform(min_val, max_val, size=nwalkers)

print(f"Starting MCMC Sampling with {nwalkers} walkers over {nsteps} steps...")
sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_probability,
    args=(bounds, target_ucs_mean, target_ucs_std, multi_gpr, scaler_X, scaler_y)
)

sampler.run_mcmc(pos, nsteps, progress=True)
print("MCMC Sampling Complete!")

print("\n STEP 4: PREDICT AND PLOT MICROPARAMETER PDFs ")
burn_in = 200
samples = sampler.get_chain(discard=burn_in, flat=True)

fig, axes = plt.subplots(5, 3, figsize=(15, 20))
axes = axes.flatten()

for i, col in enumerate(input_cols):
    sns.kdeplot(samples[:, i], ax=axes[i], fill=True, color="skyblue")
    mean_val = np.mean(samples[:, i])
    axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.4g}')
    axes[i].set_title(col, fontsize=12, weight='bold')
    axes[i].set_xlabel('Parameter Value')
    axes[i].set_ylabel('Probability Density')
    axes[i].legend()

plt.tight_layout()
plt.suptitle(f'Predicted PDFs of Microparameters (Target UCS = {target_ucs_mean/1e6:.1f} MPa)', y=1.02, fontsize=16, weight='bold')
plt.show()

best_microparameters = np.mean(samples, axis=0).reshape(1, -1)
test_scaled = scaler_X.transform(best_microparameters)
test_pred_scaled = multi_gpr.predict(test_scaled)
test_pred_orig = scaler_y.inverse_transform(test_pred_scaled).flatten()

print("\n=========================================================")
print(f"Target UCS you wanted : {target_ucs_mean / 1e6:.2f} MPa")
print(f"Predicted UCS by MCMC : {test_pred_orig[0] / 1e6:.2f} MPa")
print("=========================================================")
